# 00 - Foundations and Setup
## Shakespeare Evidence Desk

- **Goal:** prepare a reliable collection of ten Shakespeare plays in Azure so later notebooks can focus on learning RAG.
- **Not RAG yet:** this notebook does not parse the plays into passages, create embeddings, retrieve answers, or call an LLM.
- **Where it runs:** Python and the notebook kernel run on your machine. Azure stores the documents. Python invokes Azure CLI using your existing login.
- **Reading pattern:** read the Markdown explanation, run the following code cell, and check its result before moving on. Later cells use values created by earlier ones.
- **Repeatability:** reuse the existing setup by default. Four explicit switches control optional cloud changes.
- **Privacy:** subscription/resource/user identifiers come from `.env`. Generated setup files go in the ignored `.artifacts/` directory. Clear identifying outputs before sharing.

### Terms We Use

- **Subscription:** the Azure account scope under which resources are managed and billed.
- **Resource group:** a named group that organizes related Azure resources.
- **Storage account:** the Azure resource providing blob storage.
- **Container:** a collection of blobs inside the storage account. A blob is a stored file/object.
- **Corpus:** our collection of ten complete plays.
- **Manifest:** a JSON inventory of those files, their source URLs, sizes, and checksums.
- **Snapshot:** our name for a dated folder containing one corpus download; not Azure's separate blob-snapshot feature.
- **Handoff:** a saved setup report that lets the next notebook locate the verified corpus without reusing this kernel's variables.

## 1. Dependencies and Local Setup

- **Purpose:** prepare the local Python tools and file locations used by later cells. Nothing is created in Azure in this step.
- **Before running:** select the project `.venv` kernel and install dependencies in it with `python -m pip install -r requirements.txt`.
- **External commands:** Azure CLI (`az`) communicates with Azure; `curl` downloads public texts; `jq` builds JSON metadata; `shasum` calculates file fingerprints. The uploader uses the last three commands.
- **Imports:** the next cell loads Python utilities for JSON, paths, subprocess commands, temporary files, dates, checksums, and reading `.env`.
- **`ROOT`:** finds the repository folder, using `.env.example` as a marker. Starting in the wrong folder raises an error instead of loading the wrong configuration.
- **`ENV_FILE`:** points to the private `.env` file. If it is missing, copy `.env.example` to `.env` and fill in your own settings before Step 2.
- **`ARTIFACTS`:** creates the local `.artifacts/` folder if needed. This holds generated reports, not your Azure-hosted corpus.
- **Authentication prerequisite:** run interactive `az login` in your terminal for the appropriate tenant. Reading `.env` does not log you in.
- **Expected result:** 'Repository and local artifact directory ready.' No storage account, role, container, or corpus is changed.

In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from uuid import UUID

from dotenv import dotenv_values, set_key

ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')
ENV_FILE = ROOT / '.env'
ARTIFACTS = ROOT / '.artifacts'
ARTIFACTS.mkdir(exist_ok=True)
print('Repository and local artifact directory ready.')

## 2. Load and Validate Private Configuration

- **Purpose:** tell the notebook which Azure destination and user to use without hard-coding identifiers into code.
- **Input:** the private `.env` file. The checked-in `.env.example` is a template, not working configuration.
- **What the code does:** reads non-empty values into the Python `config` dictionary, checks required keys, and validates identifier/name formats before making Azure calls.
- **What it does not prove:** valid-looking names and IDs do not guarantee resources exist or that you have permission. Later steps check those against Azure.

### Configuration Values

- **`AZURE_SUBSCRIPTION_ID`:** chooses the subscription for every CLI call. The notebook does not change the global CLI default.
- **`AZURE_RESOURCE_GROUP`:** names the group containing the storage account.
- **`AZURE_LOCATION`:** chooses the region when creating missing resources; it does not move existing ones.
- **`AZURE_STORAGE_ACCOUNT`:** names the storage resource. Azure requires a globally unique name using 3-24 lowercase letters/digits.
- **`AZURE_STORAGE_CONTAINER`:** names the collection of blobs inside that account.
- **`AZURE_PRINCIPAL_ID`:** identifies the user to inspect or grant access to. It is your user object ID in the target tenant, not your email or subscription ID.
- **`AZURE_CORPUS_PREFIX`:** selects the dated corpus folder to reuse. It may be empty before the first upload, but Step 8 requires a valid selection or a new upload.
- **`AZURE_SETUP_HANDOFF_BLOB`:** optional reference written only when the setup report is uploaded in Step 10.

- **Expected result:** required configuration is validated without printing its values. If a key is missing or malformed, fix `.env` and rerun this cell.
- **Privacy:** keep `.env` private, ignored by Git, and shell-compatible. The uploader sources this trusted local file; never paste its contents into shared notebook outputs.

In [ ]:
if not ENV_FILE.is_file():
    raise RuntimeError('Create .env from .env.example and configure it first.')
config = {key: value for key, value in dotenv_values(ENV_FILE).items() if value}
required_keys = (
    'AZURE_SUBSCRIPTION_ID', 'AZURE_RESOURCE_GROUP', 'AZURE_LOCATION',
    'AZURE_STORAGE_ACCOUNT', 'AZURE_STORAGE_CONTAINER', 'AZURE_PRINCIPAL_ID',
)
missing_keys = [key for key in required_keys if not config.get(key)]
if missing_keys:
    raise RuntimeError('Missing configuration keys: ' + ', '.join(missing_keys))
UUID(config['AZURE_SUBSCRIPTION_ID'])
UUID(config['AZURE_PRINCIPAL_ID'])
if not re.fullmatch(r'[a-z0-9]{3,24}', config['AZURE_STORAGE_ACCOUNT']):
    raise ValueError('Storage account names require 3-24 lowercase letters or digits.')
container_name = config['AZURE_STORAGE_CONTAINER']
if not re.fullmatch(r'[a-z0-9][a-z0-9-]{1,61}[a-z0-9]', container_name) or '--' in container_name:
    raise ValueError('Use a valid lowercase container name without consecutive hyphens.')
print('Required configuration validated; identifiers are not displayed.')

## 3. Switches: Decide What This Run Is Allowed to Change

- **What is a switch?** A Python variable set to `True` or `False`. Later cells read it to decide whether an optional action is allowed. Setting it does not perform the action immediately.
- **`False` means:** do not perform that optional action. Checks and other enabled actions still run.
- **`True` means:** allow the relevant later cell to perform the action. It does not bypass Azure permissions or guarantee success.
- **Where to change them:** edit the four assignments in the next code cell, run that cell, then run the relevant later sections. Editing a value without rerunning the cell does not update the kernel.

### `CREATE_INFRASTRUCTURE` — Used in Steps 5 and 7

- With `False`, reuse the configured resource group, storage account, and container. Stop with an error if one is missing.
- With `True`, create only missing resources. Existing resources are reused; their settings are not silently changed.
- Enable this for a new Azure destination after reviewing names, region, permissions, and possible charges.

### `GRANT_DATA_ACCESS` — Used in Step 6

- With `False`, inspect assigned roles without adding a role.
- With `True`, grant Storage Blob Data Contributor at storage-account scope if no direct/inherited write-capable data role is found.
- The configured principal must match the signed-in user. Your identity must also have role-assignment authority; the switch cannot grant itself that authority.
- Enable this to repair missing blob access, not on every run. New permissions can take minutes to propagate.

### `UPLOAD_NEW_CORPUS` — Used in Step 8

- With `False`, reuse the snapshot selected by `AZURE_CORPUS_PREFIX` in `.env`.
- With `True`, download and upload another ten-play collection into a new dated folder, then update the selected prefix in `.env`.
- It does not overwrite or delete previous collections. Repeated enabled runs create additional copies and can incur charges.

### `UPLOAD_SETUP_HANDOFF` — Used in Step 10

- With `False`, save the setup report only in the local ignored artifact directory.
- With `True`, also upload that report to Azure and save its blob reference in `.env`.
- This uploads a small report, not another copy of the plays. The local report is sufficient for the next notebook on this machine.

### Recommended Settings

- **Your existing setup:** leave all four switches `False`; the storage, permission, and corpus are already configured.
- **Fresh setup:** enable `CREATE_INFRASTRUCTURE` and `UPLOAD_NEW_CORPUS`; enable `GRANT_DATA_ACCESS` only if a grant is needed and you are authorized. Keep Azure handoff upload optional.
- **After setup succeeds:** return the switches to `False` and rerun the switch cell before the next verification run.
- **Even with all switches `False`:** Azure read operations and sample downloads still happen, and local artifact files are written. This is read-only for cloud resources, not a no-op or a completely offline run.

In [ ]:
CREATE_INFRASTRUCTURE = False
GRANT_DATA_ACCESS = False
UPLOAD_NEW_CORPUS = False
UPLOAD_SETUP_HANDOFF = False
print('Execution switches configured. Review them before continuing.')

## 4. Azure CLI Helper and Login Check

- **Purpose:** provide one small function for later Azure commands and confirm your CLI login can access the selected subscription.
- **Dependency check:** `shutil.which('az')` checks whether Azure CLI is available on your machine. If not, install it before continuing.
- **`az_json(...)`:** takes command arguments, adds the configured subscription and JSON output options, and runs Azure CLI as a local subprocess.
- **Why JSON:** Azure's response becomes a Python dictionary, list, or other JSON value that later code can inspect instead of reading formatted terminal tables.
- **Errors:** a failing command stops execution with its diagnostic. Do not continue using incomplete results, and clear diagnostics containing resource identifiers before sharing.
- **Login check:** `account show` verifies that the selected subscription resolves correctly. This confirms access to the subscription, not yet to blob contents.
- **Authentication:** the function uses your existing Azure CLI login. It does not call `az login`, fetch account keys, or print access tokens.
- **Expected result:** 'Azure CLI can access the configured subscription.' Defining the helper itself creates no cloud resources.

In [ ]:
if shutil.which('az') is None:
    raise RuntimeError('Install Azure CLI before continuing.')

def az_json(*arguments):
    command = [
        'az', *arguments, '--subscription', config['AZURE_SUBSCRIPTION_ID'],
        '--only-show-errors', '--output', 'json',
    ]
    result = subprocess.run(command, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        raise RuntimeError(
            'Azure CLI failed. Check login, permissions, and role propagation. '
            'Clear diagnostic outputs before sharing.\n' + result.stderr.strip()
        )
    return json.loads(result.stdout) if result.stdout.strip() else None

subscription = az_json('account', 'show')
if subscription['id'].lower() != config['AZURE_SUBSCRIPTION_ID'].lower():
    raise RuntimeError('CLI did not resolve the configured subscription.')
print('Azure CLI can access the configured subscription.')

## 5. Create or Reuse the Resource Group and Storage Account

- **Purpose:** ensure the Azure storage destination exists and follows our initial security choices. This does not upload the plays or create the container yet.
- **First check:** ask Azure whether the configured resource group exists. It organizes the storage account and other related resources.
- **Second check:** look for the configured storage account inside that group.
- **With `CREATE_INFRASTRUCTURE = False`:** reuse existing resources. If the group or account is missing, stop rather than create something unexpectedly.
- **With `CREATE_INFRASTRUCTURE = True`:** create only missing resources using the names and region from `.env`. Existing account settings are inspected, not silently rewritten.

### Initial Storage Choices

- **StorageV2 / Standard:** a general-purpose storage account suitable for storing our text files and generated artifacts.
- **LRS:** keeps redundant copies within one location. It is a cost-conscious learning choice, not protection against a regional disaster.
- **Hot access:** suits files we expect to read during repeated experiments. Production requirements may justify different settings.
- **HTTPS and minimum TLS 1.2:** require encrypted transport to the storage service.
- **Anonymous blob access disabled:** unauthenticated visitors cannot read the files.
- **Shared keys disabled:** blob access uses Microsoft Entra identity rather than storage-account keys.

- **Values saved for later:** `storage_scope` identifies the account for a narrowly scoped role grant; `blob_endpoint` is the service address used in the setup report.
- **Expected result:** the account exists and its checked security settings match the baseline. If an existing account differs, the code asks you to review it instead of changing it automatically.

In [ ]:
resource_group = config['AZURE_RESOURCE_GROUP']
group_exists = az_json('group', 'exists', '--name', resource_group)
if not group_exists:
    if not CREATE_INFRASTRUCTURE:
        raise RuntimeError('Group missing. Enable CREATE_INFRASTRUCTURE if creation is approved.')
    az_json('group', 'create', '--name', resource_group, '--location', config['AZURE_LOCATION'])
accounts = az_json('storage', 'account', 'list', '--resource-group', resource_group)
account = next((item for item in accounts if item['name'] == config['AZURE_STORAGE_ACCOUNT']), None)
if account is None:
    if not CREATE_INFRASTRUCTURE:
        raise RuntimeError('Account missing. Enable CREATE_INFRASTRUCTURE if creation is approved.')
    account = az_json(
        'storage', 'account', 'create', '--resource-group', resource_group,
        '--name', config['AZURE_STORAGE_ACCOUNT'], '--location', config['AZURE_LOCATION'],
        '--kind', 'StorageV2', '--sku', 'Standard_LRS', '--access-tier', 'Hot',
        '--https-only', 'true', '--min-tls-version', 'TLS1_2',
        '--allow-blob-public-access', 'false', '--allow-shared-key-access', 'false',
    )
account = az_json(
    'storage', 'account', 'show', '--resource-group', resource_group,
    '--name', config['AZURE_STORAGE_ACCOUNT'],
)
if account.get('allowBlobPublicAccess') is not False or account.get('allowSharedKeyAccess') is not False:
    raise RuntimeError('Review the existing account: expected anonymous access and shared keys to be disabled.')
if account.get('enableHttpsTrafficOnly') is not True or account.get('minimumTlsVersion') not in ('TLS1_2', 'TLS1_3'):
    raise RuntimeError('Review the existing account HTTPS and minimum TLS settings.')
storage_scope = account['id']
blob_endpoint = account['primaryEndpoints']['blob'].rstrip('/')
print('Storage account verified: anonymous access and shared keys are disabled.')

## 6. Diagnose and Optionally Fix Blob Permissions

- **Purpose:** address the portal error saying your Microsoft Entra user cannot list blob data.
- **Key distinction:** managing a storage account and accessing its files are different permissions. Contributor can manage the resource but does not automatically grant blob read/write access.
- **Inspect first:** list the configured user's direct and inherited role assignments at the storage-account scope. Look for Storage Blob Data Contributor or Storage Blob Data Owner.
- **With `GRANT_DATA_ACCESS = False`:** report what was found without adding a role. This is the setting to use when access is already configured.
- **With `GRANT_DATA_ACCESS = True`:** if no listed write-capable data role is found, check that the configured principal matches your signed-in user, then request Storage Blob Data Contributor.
- **What that role allows:** reading and uploading blob data, along with other blob data operations. The grant is limited to this storage account, not the entire subscription.
- **Required authority:** you must already be authorized to assign roles, for example through User Access Administrator or Owner. The switch cannot manufacture this authority.
- **Important limits:** group-based access, custom roles, and privileged-role activation can affect effective permissions. The role list is a diagnostic; Step 7 tests actual access.
- **Expected result:** an existing role is reported, a missing role is granted when enabled and authorized, or the notebook explains that access still needs attention.
- **After a new grant:** allow minutes for propagation and refresh the portal using the same tenant/user. Do not enable keys or anonymous access to work around the error.

In [ ]:
assignments = az_json(
    'role', 'assignment', 'list', '--assignee', config['AZURE_PRINCIPAL_ID'],
    '--scope', storage_scope, '--include-inherited',
)
write_roles = {'Storage Blob Data Contributor', 'Storage Blob Data Owner'}
has_write_role = any(item.get('roleDefinitionName') in write_roles for item in assignments)
if GRANT_DATA_ACCESS and not has_write_role:
    signed_in_user = az_json('ad', 'signed-in-user', 'show', '--query', 'id')
    if signed_in_user.lower() != config['AZURE_PRINCIPAL_ID'].lower():
        raise RuntimeError('Configured principal differs from the signed-in user; review .env before granting access.')
    az_json(
        'role', 'assignment', 'create', '--assignee-object-id', config['AZURE_PRINCIPAL_ID'],
        '--assignee-principal-type', 'User', '--role', 'Storage Blob Data Contributor',
        '--scope', storage_scope,
    )
    print('Data role granted at account scope. Allow propagation before testing access.')
elif has_write_role:
    print('A direct/inherited write-capable blob data role is already assigned.')
else:
    print('No direct/inherited write role found. Test access next; request help from an authorized role administrator if needed.')

## 7. Verify Access and Create a Private Container

- **Purpose:** confirm that Azure actually lets your signed-in identity access blob storage, then prepare the collection that will hold the plays.
- **Container meaning:** a named collection of blobs inside the storage account. Each uploaded play becomes a blob inside this collection.
- **Authentication:** every storage command uses `--auth-mode login`, meaning your Azure CLI identity rather than an account key.
- **Existence check:** ask whether the configured container already exists. This is a real data-access operation, unlike just inspecting the storage account's management settings.
- **With `CREATE_INFRASTRUCTURE = False`:** reuse an existing container; stop if it is missing.
- **With `CREATE_INFRASTRUCTURE = True`:** create a missing container with anonymous access disabled. This still requires effective data permissions.
- **Privacy check:** inspect the resulting container and reject an anonymous-access setting.
- **Network distinction:** private here means no anonymous access. It does not mean we created a private network endpoint; authenticated access to the public service endpoint is used initially.
- **Expected result:** data access succeeds and the configured container is private. No plays are uploaded in this step.
- **If it fails after Step 6:** allow role propagation, verify the signed-in tenant/user, and rerun this check. A failure should not trigger a less secure workaround.

In [ ]:
container_exists = az_json(
    'storage', 'container', 'exists', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
    '--name', config['AZURE_STORAGE_CONTAINER'], '--auth-mode', 'login',
)['exists']
if not container_exists:
    if not CREATE_INFRASTRUCTURE:
        raise RuntimeError('Container missing. Enable CREATE_INFRASTRUCTURE if creation is approved.')
    az_json(
        'storage', 'container', 'create', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
        '--name', config['AZURE_STORAGE_CONTAINER'], '--auth-mode', 'login', '--public-access', 'off',
    )
container_properties = az_json(
    'storage', 'container', 'show', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
    '--name', config['AZURE_STORAGE_CONTAINER'], '--auth-mode', 'login',
)
if container_properties.get('properties', {}).get('publicAccess') not in (None, 'None'):
    raise RuntimeError('Container is not private; review anonymous-access settings.')
print('Blob data access succeeded; the configured container is private.')

## 8. Upload the Corpus or Reuse a Snapshot

- **Purpose:** select the ten-play document collection that later RAG stages will use. The corpus is the documents themselves, not embeddings or an index.
- **With `UPLOAD_NEW_CORPUS = False`:** do not download or upload another collection. Reuse the dated folder named by `AZURE_CORPUS_PREFIX` in `.env`. Step 9 checks its contents.
- **With `UPLOAD_NEW_CORPUS = True`:** run `scripts/upload_shakespeare.sh`, which reads the same `.env` and downloads ten complete Shakespeare plays from Project Gutenberg.
- **Before upload:** the script checks each title/author and preserves the original file, including its Gutenberg notices.
- **Metadata:** the script creates a manifest containing source URLs, titles, file sizes, and SHA-256 checksums. This records where each document came from and how to verify it.
- **Storage layout:** upload ten text files and one manifest into a new dated folder. Earlier folders are not overwritten or deleted; repeated new uploads create additional copies.
- **Verification inside the script:** check that eleven blobs exist and that downloading Hamlet returns identical bytes.
- **Selection update:** read the script's local JSON result and update only `AZURE_CORPUS_PREFIX` in `.env`, so future runs use the new collection.
- **Expected result:** a corpus folder is selected. With uploading disabled, an empty or placeholder prefix raises an error instead of guessing a collection.
- **Rights and later processing:** original Shakespeare works are public domain; verify edition terms and applicable local law. Later parsing excludes administrative notices from searchable content while preserving unchanged originals.
- **Current limitation:** these editions are text-only. Scanned or illustrated editions are a separate future experiment, not something this upload invents.

In [ ]:
if UPLOAD_NEW_CORPUS:
    result = subprocess.run(
        ['bash', str(ROOT / 'scripts' / 'upload_shakespeare.sh')],
        cwd=ROOT, capture_output=True, text=True, timeout=900,
    )
    if result.returncode != 0:
        raise RuntimeError('Corpus upload failed.\n' + result.stderr.strip())
    upload_result = json.loads((ARTIFACTS / 'corpus-upload.json').read_text(encoding='utf-8'))
    if upload_result['document_count'] != 10:
        raise RuntimeError('Uploader did not produce the expected ten-document result.')
    config['AZURE_CORPUS_PREFIX'] = upload_result['prefix']
    set_key(str(ENV_FILE), 'AZURE_CORPUS_PREFIX', upload_result['prefix'], quote_mode='always')
    print('New snapshot uploaded and selected in .env.')
corpus_prefix = config.get('AZURE_CORPUS_PREFIX', '').strip('/')
if not corpus_prefix or corpus_prefix.endswith('your-snapshot'):
    raise RuntimeError('Configure AZURE_CORPUS_PREFIX or enable UPLOAD_NEW_CORPUS.')
print('Snapshot selected; its location remains in private configuration.')

## 9. Validate Inventory and a Download Checksum

- **Purpose:** prove that the selected corpus is present and consistent before a later stage processes it. A successful upload message alone is not enough.
- **Read the manifest:** download its JSON into a temporary local folder and read the ten document entries.
- **Check document identities:** require the expected ten distinct Gutenberg works and the author William Shakespeare.
- **Check metadata:** ensure each document refers to the selected folder, has a positive byte count, and has a correctly formatted SHA-256 checksum.
- **Compare with Azure:** list blobs in that folder. Their names must match the ten texts plus the manifest, and each text's size must match the inventory.
- **What is a checksum?** A fingerprint calculated from file contents. Different bytes are overwhelmingly likely to produce a different SHA-256 fingerprint.
- **Verify Hamlet:** download its actual contents, calculate SHA-256 locally, and compare that value with the manifest. This checks Hamlet's full contents; it does not fully hash all ten uploaded files.
- **Local cleanup:** verification downloads use a temporary directory that is removed when the check completes.
- **Expected result:** ten titles/sizes and a confirmation that ten plays, one manifest, and Hamlet's checksum were verified. Azure identifiers are not intentionally printed.
- **What this does not evaluate:** literary answers, parsing accuracy, retrieval quality, or model performance. Those require later experiments.

In [ ]:
def download_blob(blob_name, file_path):
    return az_json(
        'storage', 'blob', 'download', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
        '--container-name', config['AZURE_STORAGE_CONTAINER'], '--name', blob_name,
        '--file', str(file_path), '--auth-mode', 'login',
    )

manifest_blob = corpus_prefix + '/manifest.json'
with tempfile.TemporaryDirectory() as directory:
    temporary_path = Path(directory)
    manifest_path = temporary_path / 'manifest.json'
    download_blob(manifest_blob, manifest_path)
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    documents = manifest['documents']
    expected_ids = {f'gutenberg-{ebook_id}' for ebook_id in (1524, 1533, 1513, 1531, 1532, 1514, 1515, 1522, 1540, 1526)}
    if len(documents) != 10 or {document['id'] for document in documents} != expected_ids:
        raise RuntimeError('Manifest does not contain the expected ten distinct plays.')
    for document in documents:
        if document['author'] != 'William Shakespeare' or not document['original_notices_preserved']:
            raise RuntimeError('Unexpected author or missing original-notices flag.')
        if not document['blob_name'].startswith(corpus_prefix + '/'):
            raise RuntimeError('Manifest points outside the selected snapshot.')
        if document['bytes'] <= 0 or not re.fullmatch(r'[0-9a-f]{64}', document['sha256']):
            raise RuntimeError('Invalid document size or checksum metadata.')
    blobs = az_json(
        'storage', 'blob', 'list', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
        '--container-name', config['AZURE_STORAGE_CONTAINER'], '--prefix', corpus_prefix + '/',
        '--auth-mode', 'login',
    )
    blob_sizes = {blob['name']: blob['properties']['contentLength'] for blob in blobs}
    expected_names = {document['blob_name'] for document in documents} | {manifest_blob}
    if set(blob_sizes) != expected_names:
        raise RuntimeError('Inventory does not match the ten texts and manifest.')
    if any(blob_sizes[document['blob_name']] != document['bytes'] for document in documents):
        raise RuntimeError('Document sizes do not match the manifest.')
    hamlet = next(document for document in documents if document['id'] == 'gutenberg-1524')
    hamlet_path = temporary_path / 'hamlet.txt'
    download_blob(hamlet['blob_name'], hamlet_path)
    if hashlib.sha256(hamlet_path.read_bytes()).hexdigest() != hamlet['sha256']:
        raise RuntimeError('Hamlet checksum verification failed.')
for document in documents:
    print(f"{document['title']}: {document['bytes']:,} bytes")
print('Verified ten plays, one manifest, and the Hamlet checksum.')

## 10. Save a Setup Report for the Next Notebook

- **Purpose:** save the information the next notebook needs to locate our verified documents. A handoff is just this saved setup report, not a new Azure service.
- **Why save it?** Python variables disappear when the kernel restarts. A JSON file survives and can be read independently by the ingestion notebook.
- **This is not another corpus copy:** the report contains references and metadata, not the complete text of the plays. It does not parse, embed, or index documents, and does not call an LLM.

### What the `handoff` Dictionary Contains

- **`stage` and `created_at`:** identify which setup step produced the report and when it ran.
- **`storage`:** records the selected subscription, resource group, storage account, container, and blob service address. Values come from private configuration and verified Azure results.
- **`corpus_prefix` and `source_manifest_blob`:** identify the exact corpus folder and its manifest, rather than asking the next notebook to guess the newest folder.
- **`documents`:** copies the manifest's file references, source information, sizes, and checksums into the report.
- **`verification`:** records the checks completed in Step 9: ten documents, eleven blobs, and the verified Hamlet checksum. It is a setup result, not a RAG quality score.
- **`decision`:** explains the initial choices and the next experiment. Ten related plays support cross-work questions; identity authentication avoids shared keys; Standard LRS is an initial learning choice to revisit for production resilience.

### What Gets Written

- **Always, after earlier steps succeed:** save `.artifacts/setup-handoff.json` locally. Rerunning this step replaces that local report with the current verified setup.
- **How the save works:** `json.dumps(handoff, indent=2)` converts the Python dictionary into readable JSON text; `write_text(...)` writes that text into the file.
- **With `UPLOAD_SETUP_HANDOFF = False`:** stop at the local report. Nothing is uploaded by this step, and `.env` is not changed.
- **With `UPLOAD_SETUP_HANDOFF = True`:** also upload the report to a new Azure blob and record its reference as `AZURE_SETUP_HANDOFF_BLOB` in `.env`. It does not upload the plays again.
- **Recommended now:** leave the upload switch `False`. The next notebook on this machine can read the local report.
- **Expected result:** 'Local handoff saved in .artifacts/setup-handoff.json.' We can now restart the kernel without losing the information needed to find the corpus.
- **Privacy:** the report contains your Azure identifiers and is ignored by Git. Do not publish it or identifying notebook outputs; a cloud copy remains in the private container.

In [ ]:
handoff = {
    'stage': '00-foundations-storage',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'storage': {
        'subscription_id': config['AZURE_SUBSCRIPTION_ID'],
        'resource_group': config['AZURE_RESOURCE_GROUP'],
        'account': config['AZURE_STORAGE_ACCOUNT'],
        'container': config['AZURE_STORAGE_CONTAINER'],
        'blob_endpoint': blob_endpoint,
    },
    'corpus_prefix': corpus_prefix,
    'source_manifest_blob': manifest_blob,
    'documents': documents,
    'verification': {'document_count': len(documents), 'blob_count': len(blobs), 'hamlet_sha256_verified': True},
    'decision': {
        'corpus': 'Ten related Shakespeare plays support passage and cross-work questions.',
        'authentication': 'Microsoft Entra identity; no account keys or anonymous blob access.',
        'storage': 'Standard LRS is the learning baseline; review production resilience separately.',
        'next_stage': 'Compare deterministic and model-assisted parsing while preserving provenance.',
    },
}
handoff_path = ARTIFACTS / 'setup-handoff.json'
handoff_path.write_text(json.dumps(handoff, indent=2) + '\n', encoding='utf-8')
if UPLOAD_SETUP_HANDOFF:
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    handoff_blob = f'setup/{run_id}/handoff.json'
    az_json(
        'storage', 'blob', 'upload', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
        '--container-name', config['AZURE_STORAGE_CONTAINER'], '--name', handoff_blob,
        '--file', str(handoff_path), '--auth-mode', 'login', '--overwrite', 'false',
    )
    set_key(str(ENV_FILE), 'AZURE_SETUP_HANDOFF_BLOB', handoff_blob, quote_mode='always')
    print('Setup handoff uploaded; its reference is stored in .env.')
print('Local handoff saved in .artifacts/setup-handoff.json.')

## Completion Check and Next Sections

- **For your current setup:** keep all four switches `False`. Restart the kernel and run the code cells in order to confirm the notebook does not depend on hidden state.
- **Successful setup means:** subscription access works, the account passes security checks, the container is private and accessible, ten plays and their manifest are present, Hamlet's checksum matches, and the local handoff exists.
- **After granting a role:** allow propagation and rerun the data-access check. Do not upload again just to fix an authorization error.
- **Before sharing:** clear notebook outputs. `.env`, `.venv/`, and `.artifacts/` are ignored. Share the placeholder `.env.example`, not your configured file or identifying generated reports.
- **What we have achieved:** a known, accessible, verified document collection. No embeddings, search index, or RAG answers have been created yet.
- **Next foundations sections:** connect hosted chat/embedding models, add telemetry, and build labeled development/held-out questions.
- **Next stage notebook:** load the setup report, compare parsing techniques on the selected plays, inspect metrics and errors, justify a choice, and save its outputs for chunking.